# Final Report: CORN ETF Trading Signal Pipeline

Generated at `2026-05-28 02:58:49 UTC`.

This report is generated by `uv run --extra dev doit docs`. It consolidates the code map, data inventory, main results, auxiliary results, figures, and known problems for the CORN ETF weekly trading-signal project.


## Open The HTML Outputs

```bash
open reports/chartbook/index.html
open reports/html/corn_forecast_workflow.html
```


## Research Design In One Paragraph

The project treats CORN ETF forecasting as a weekly trading-signal problem. The main target is a fixed 2 percent next-week return band:

```text
Y =  1 if next_week_return >= +2%
Y =  0 if -2% < next_week_return < +2%
Y = -1 if next_week_return <= -2%
```

The main comparison is `price_only` versus `price_calendar` under expanding walk-forward validation after `2022-12-31`. Auxiliary checks include expected-return trading, volatility-adjusted target selection, and a secondary binary up/down pipeline.


In [ ]:
# Executive summary


item,result
Frozen sample,"2011-01-07 to 2026-05-15, 802 weekly rows"
OOS window,"2023-01-06 to 2026-05-08, 175 weeks / 1400 prediction rows"
Main fixed-band classifier,"price_ai: balanced acc 32.6%, macro F1 32.6%"
Best expected-return strategy,"price_calendar_ai_ridge: return 10.7%, Sharpe 0.322"
Best volatility-threshold strategy,not generated
Secondary binary direction result,not generated


In [ ]:
# OOS fixed-band class distribution


OOS fixed-band class distribution

In [ ]:
# Strategy Sharpe comparison


Strategy Sharpe comparison

In [ ]:
# Main fixed 2 percent classification result


features,accuracy,balanced acc,macro F1,down,flat,up,OOS rows,folds
price_ai,49.7%,32.6%,32.6%,32,119,24,175,14
price_only,46.9%,31.9%,31.4%,32,119,24,175,14
price_calendar,42.3%,30.9%,30.4%,32,119,24,175,14
price_calendar_ai,42.9%,30.0%,30.0%,32,119,24,175,14


In [ ]:
# Return-regression diagnostic inside target test


features,MAE,RMSE,R2,direction acc,OOS rows,folds
price_calendar_ai,0.0180,0.0252,-0.1018,53.7%,175,14
price_calendar,0.0173,0.0243,-0.0220,53.1%,175,14
price_only,0.0175,0.0246,-0.0449,52.0%,175,14
price_ai,0.0182,0.0254,-0.1161,48.6%,175,14


In [ ]:
# Auxiliary expected-return strategy


run,features,model,MAE,RMSE,R2,direction acc,trade freq,strategy return,Sharpe,max drawdown
price_calendar_ai_ridge,price_calendar_ai,ridge,0.0180,0.0252,-0.1018,53.7%,27.4%,10.7%,0.322,-14.8%
price_calendar_hgb,price_calendar,hgb,0.0181,0.0252,-0.0989,53.7%,30.9%,3.6%,0.130,-9.4%
price_only_ridge,price_only,ridge,0.0175,0.0246,-0.0449,52.0%,11.4%,3.3%,0.089,-14.2%
price_ai_ridge,price_ai,ridge,0.0182,0.0254,-0.1161,48.6%,20.6%,3.1%,0.077,-20.7%
price_calendar_ridge,price_calendar,ridge,0.0173,0.0243,-0.0220,53.1%,21.1%,-6.2%,-0.188,-16.8%
price_calendar_ai_hgb,price_calendar_ai,hgb,0.0182,0.0247,-0.0527,53.1%,33.7%,-5.8%,-0.208,-13.6%
price_only_hgb,price_only,hgb,0.0187,0.0258,-0.1535,50.9%,32.0%,-25.9%,-0.769,-32.8%
price_ai_hgb,price_ai,hgb,0.0189,0.0259,-0.1640,46.9%,30.9%,-30.9%,-0.997,-33.6%


In [ ]:
# Volatility-adjusted threshold check


Volatility-adjusted threshold check

In [ ]:
# Secondary binary direction pipeline


Secondary binary direction pipeline

In [ ]:
# Feature panel summary


rows,cols,first_week,last_week,price_features,calendar_features,weather_features,text_features,ai_features
802,33,2011-01-07,2026-05-15,11,9,0,0,7


In [ ]:
# Data and report artifacts


artifact,exists,rows,cols
data/raw/prices_CORN.csv,yes,3864,7
data/raw/usda_releases.csv,no,,
data/interim/weather_weekly.parquet,no,,
data/interim/ai_weekly.parquet,yes,789,8
data/processed/feature_panel.parquet,yes,802,33
reports/price_target_predictions.csv,yes,1400,13
reports/expected_return_predictions.csv,yes,1400,18
reports/threshold_selection_predictions.csv,no,,
reports/predictions.csv,no,,


In [ ]:
# Code coverage map


area,files,role
Task graph,dodo.py,"Single pydoit entrypoint for data, models, reports, ChartBook, tests."
CLI orchestration,src/corn_forecast/cli.py,Command surface used by pydoit; now reuses cached prices when available.
Configuration,src/corn_forecast/config.py,"Research defaults, dates, paths, thresholds, feature-set names."
Data adapters,"data/prices.py, data/weather.py, data/usda.py","Price pulls, weather cache/demo adapter, USDA text adapter."
Feature panel,src/corn_forecast/features.py,"Weekly price, calendar, weather, text, AI feature joins."
Main target test,src/corn_forecast/price_target_tests.py,Fixed 2 percent three-class target plus return-regression diagnostics.
Return strategy,src/corn_forecast/expected_return_strategy.py,"Expected-return models, trading threshold, strategy returns."
Threshold robustness,src/corn_forecast/threshold_selection.py,Volatility-adjusted 3-class target selection.
Binary direction,"src/corn_forecast/models.py, strategy.py",Secondary up/down classifier and trading backtest.
WWCB/AI text,"src/corn_forecast/text/*.py, scripts/*.py",PDF download/parse and GLM/mock structured AI features.


In [ ]:
# pydoit task map


task,purpose,main outputs
baseline,Current fixed 2 percent classification baseline,"price_target_tests.json, price_target_predictions.csv"
research,Main experiment bundle,"classification, expected-return, threshold, notebook"
core,Feature panel plus binary direction report,"feature_panel.parquet, metrics.json, figures"
docs,"Final report, notebook, ChartBook","reports/html, reports/notebooks, reports/chartbook"
refresh_data,Explicit network refresh,"Yahoo prices, USDA releases, weather cache/catalog"
wwcb_*,Optional text/AI feature pipeline,"WWCB manifest, parsed text, ai_weekly.parquet"
tests,Project test suite,37 tests


## Problems And Caveats

- The fixed 2 percent target is imbalanced: 119 of 175 OOS weeks are flat. Accuracy alone is not enough.
- Calendar seasonality does not improve the main fixed-band classifier in this run; balanced accuracy is 1.0% lower than price-only.
- All expected-return regressions have negative OOS R2, so trading metrics should be treated as fragile.
- Volatility-threshold selection outputs are missing; run `uv run --extra dev doit select_threshold`.
- Optional weather/text/AI data exist locally, but the main default feature sets are still `price_only` and `price_calendar`.
- Generated report and data outputs are ignored by git; rerun `uv run --extra dev doit docs` before submission.


In [ ]:
from pathlib import Path
import pandas as pd

root = Path("..").resolve().parent
feature_panel = pd.read_parquet(root / "data/processed/feature_panel.parquet")
price_metrics = pd.read_json(root / "reports/price_target_tests.json").T
expected_return_metrics = pd.read_json(root / "reports/expected_return_metrics.json").T
threshold_metrics = pd.read_json(root / "reports/threshold_selection.json").T
feature_panel.tail()


## Final Report Figures


### Final Class Distribution

![Final Class Distribution](../figures/final_class_distribution.png)


### Final Strategy Sharpe

![Final Strategy Sharpe](../figures/final_strategy_sharpe.png)


### Final Strategy Return

![Final Strategy Return](../figures/final_strategy_return.png)


### Final Expected Return Cumulative

![Final Expected Return Cumulative](../figures/final_expected_return_cumulative.png)


### Final Fixed Target Confusion

![Final Fixed Target Confusion](../figures/final_fixed_target_confusion.png)


## Generated Outputs

```text
reports/notebooks/corn_forecast_workflow.ipynb
reports/html/corn_forecast_workflow.html
reports/chartbook/index.html
```

Open the HTML outputs on macOS:

```bash
open reports/chartbook/index.html
open reports/html/corn_forecast_workflow.html
```

The notebook and HTML report are generated artifacts. Source documentation lives in `docs_src/project_workflow.md`; ChartBook configuration lives in `chartbook.toml`; workflow automation lives in `dodo.py`.
